In [ ]:
import pandas as pd
import numpy as np
import os

# --- Configuration ---
input_file = "CIC-ToN-IoT.csv"
temp_cleaned_file = "cleaned_data_temp.csv"
chunk_size = 100000  # Adjust this based on your RAM (100k is usually safe)

columns_to_keep = [
    'Src IP', 'Dst IP', 'Timestamp', 'Label', 
    'Src Port', 'Dst Port', 'Fwd Header Len', 'Init Bwd Win Byts', 
    'Fwd Seg Size Avg', 'Fwd Pkt Len Mean', 'Init Fwd Win Byts', 
    'Fwd Pkt Len Max', 'TotLen Fwd Pkts', 'Bwd Pkt Len Mean', 
    'Idle Min', 'Bwd Header Len', 'Pkt Len Var', 'Subflow Fwd Byts', 
    'TotLen Bwd Pkts', 'Idle Max', 'Fwd Seg Size Min', 'Idle Mean', 
    'Pkt Len Max', 'Bwd Pkt Len Std', 'Bwd Pkt Len Max', 
    'Protocol', 'Pkt Len Mean', 'Down/Up Ratio'
]

# 1. Process in Chunks
print(f"Processing {input_file} in batches of {chunk_size}...")

# Initialize the CSV file (write header only)
first_chunk = True

for chunk in pd.read_csv(input_file, chunksize=chunk_size, low_memory=False):
    # Clean column names
    chunk.columns = chunk.columns.str.strip()
    
    # Select only required columns
    chunk = chunk[columns_to_keep]
    
    # Handle Infinity and NaNs
    chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
    chunk.dropna(inplace=True)
    
    # Parse Timestamps
    chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')
    chunk.dropna(subset=['Timestamp'], inplace=True)
    
    # Standardize Labels
    chunk['Label'] = chunk['Label'].apply(lambda x: 0 if str(x).strip() in ['Normal', 'Benign', '0', 0] else 1)
    
    # Append to temporary file
    if first_chunk:
        chunk.to_csv(temp_cleaned_file, index=False, mode='w')
        first_chunk = False
    else:
        chunk.to_csv(temp_cleaned_file, index=False, mode='a', header=False)

# 2. Final Sorting and Splitting
print("Loading cleaned data for sorting and splitting...")
# Now that the file is filtered (smaller), we can try to load it for the final steps
df = pd.read_csv(temp_cleaned_file)

print("Sorting by Timestamp...")
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values(by='Timestamp', ascending=True)

# 3. Split the Data
print("Creating Train/Test sets...")
benign_data = df[df['Label'] == 0]
attack_data = df[df['Label'] == 1]

train_size = 100000 
train_data = benign_data.head(train_size)

test_benign = benign_data.iloc[train_size : train_size + 50000] 
test_attacks = attack_data.head(50000)

test_data = pd.concat([test_benign, test_attacks])
test_data = test_data.sort_values(by='Timestamp', ascending=True)

# 4. Save Final Datasets
print("Saving train_data.csv and test_data.csv...")
train_data.to_csv("train_data.csv", index=False)
test_data.to_csv("test_data.csv", index=False)

# Optional: Remove the temporary file
if os.path.exists(temp_cleaned_file):
    os.remove(temp_cleaned_file)

print("Preprocessing complete!")

Loading raw dataset...
